# Step 1: Exploratory Data Analysis & Advanced Feature Engineering (Optimized)
**Titanic Disaster Survival Prediction**

This notebook performs data analysis, feature engineering, and robust missing value imputation.

> **Constraint Enforcement**:  is strictly isolated during EDA and feature exploration.

## 1. Data Loading & Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 7)

raw_df = pd.read_csv('data/train.csv')
print(f'Loaded train dataset with shape: {raw_df.shape}')
raw_df.head()

## 2. Feature Dependency & Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Sex
raw_df.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[0, 0], color=['salmon', 'lightblue'])
axes[0, 0].set_title('Survival Rate by Sex')

# 2. Pclass
raw_df.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[0, 1], color=['gold', 'silver', 'darkgoldenrod'])
axes[0, 1].set_title('Survival Rate by Pclass')

# 3. Pclass x Sex
raw_df.groupby(['Pclass', 'Sex'])['Survived'].mean().unstack().plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Survival Rate by Pclass & Sex')

# 4. Embarked
raw_df.groupby('Embarked')['Survived'].mean().plot(kind='bar', ax=axes[1, 1], color='teal')
axes[1, 1].set_title('Survival Rate by Embarked Port')

plt.tight_layout()
plt.savefig('eda_target_dependencies.png', dpi=300)
plt.show()

## 3. Advanced Feature Engineering & WCG Signal

In [ ]:
df = raw_df.copy()

title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Mlle': 'Miss', 'Mme': 'Mrs', 'Ms': 'Miss',
    'Lady': 'Noble', 'Countess': 'Noble', 'Sir': 'Noble', 'Don': 'Noble', 'Jonkheer': 'Noble',
    'Capt': 'Officer', 'Col': 'Officer', 'Major': 'Officer', 'Dr': 'Officer', 'Rev': 'Rev'
}

df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['TitleGroup'] = df['Title'].map(title_mapping).fillna('Rare')
df['IsWomanChild'] = ((df['TitleGroup'] == 'Master') | (df['Sex'] == 'female')).astype(int)
df['Surname'] = df['Name'].str.split(',').str[0]

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['FamilyType'] = pd.cut(df['FamilySize'], bins=[0, 1, 4, 20], labels=['Single', 'Small', 'Large'])

df['CabinDeck'] = df['Cabin'].str[0].fillna('U')
df['HasCabin'] = (df['Cabin'].notnull()).astype(int)

# Shared ticket group size
ticket_counts = df['Ticket'].value_counts()
df['TicketGroupSize'] = df['Ticket'].map(ticket_counts)

# Fare & Age Imputation
df['Fare'] = df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize']
df['LogFarePerPerson'] = np.log1p(df['FarePerPerson'])
df['Age'] = df.groupby(['TitleGroup', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
df['Embarked'] = df['Embarked'].fillna('S')
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 50, 100], labels=['Child', 'Teen', 'Adult', 'Senior'])

# WCG Survival Rate (Calculated Out-of-Fold / Excluding Self)
train_wcg = df[df['IsWomanChild'] == 1]
ticket_wcg_count = train_wcg.groupby('Ticket')['PassengerId'].count()
ticket_wcg_survived = train_wcg.groupby('Ticket')['Survived'].sum()
surname_wcg_count = train_wcg.groupby(['Surname', 'Pclass'])['PassengerId'].count()
surname_wcg_survived = train_wcg.groupby(['Surname', 'Pclass'])['Survived'].sum()

def compute_wcg_train(row):
    ticket = row['Ticket']
    surname = row['Surname']
    pclass = row['Pclass']
    is_wc = row['IsWomanChild']
    
    t_count = ticket_wcg_count.get(ticket, 0)
    t_surv = ticket_wcg_survived.get(ticket, 0)
    if is_wc == 1:
        t_count -= 1
        t_surv -= row['Survived']
    if t_count > 0:
        return t_surv / t_count
        
    s_count = surname_wcg_count.get((surname, pclass), 0)
    s_surv = surname_wcg_survived.get((surname, pclass), 0)
    if is_wc == 1:
        s_count -= 1
        s_surv -= row['Survived']
    if s_count > 0:
        return s_surv / s_count
        
    return -1.0

df['WCG_Rate'] = df.apply(compute_wcg_train, axis=1)

print('Feature Engineering & Corrected WCG Rate Complete!')
print(df[['PassengerId', 'Name', 'TitleGroup', 'FamilyType', 'FarePerPerson', 'WCG_Rate']].head(10))

## 4. Save Cleaned Training Data

In [ ]:
df.to_csv('data/train_cleaned.csv', index=False)
print('Successfully saved data/train_cleaned.csv!')